# Perbandingan Pendekatan *Lexical* dan *Semantic* pada *Product Search*
### Menggunakan Amazon Shopping Queries ESCI Dataset: Analisis Kinerja dan Karakteristik Hasil Klasifikasi

**Nama:** Ryan &nbsp;|&nbsp; **NIM:** 23051204205 &nbsp;|&nbsp; **Program Studi:** S1 Teknik Informatika, Universitas Negeri Surabaya
**Pembimbing:** Anita Qoiriah, S.Kom., M.Kom.

---

## Ringkasan Penelitian

Penelitian ini membandingkan dua paradigma pemodelan relevansi *query*-produk pada domain *e-commerce search*:

1. **Pendekatan Lexical** — TF-IDF (*term frequency-inverse document frequency*) sebagai representasi fitur, dipasangkan dengan **Linear Support Vector Machine (SVM)** sebagai classifier. Direpresentasikan dalam dua varian:
   - *Separate*: query dan product title masing-masing di-vectorize sendiri lalu digabung (concat fitur).
   - *Concatenated*: query dan product title digabung dulu sebagai satu string, baru di-vectorize bersama.
2. **Pendekatan Semantic** — *fine-tuning* model *pretrained transformer* **DeBERTa-v3-base** untuk klasifikasi pasangan (query, product title).

Tugas klasifikasi adalah **4 kelas relevansi ESCI**:

| Label | Nama | Deskripsi singkat |
|---|---|---|
| **E** | Exact | Produk sepenuhnya relevan dengan query |
| **S** | Substitute | Produk mirip/pengganti, tidak 100% sesuai |
| **C** | Complement | Produk pelengkap, bukan yang dicari langsung |
| **I** | Irrelevant | Produk tidak relevan sama sekali |

## Ruang Lingkup Penelitian (Batasan Masalah)

Poin-poin berikut **dikunci di muka** (bukan hasil eksplorasi coba-coba saat menjalankan eksperimen), agar tidak ada *p-hacking* ataupun keputusan metodologis yang dibuat setelah melihat hasil test:

- **Data**: hanya `product_locale == "us"` dan `large_version == 1` dari dataset ESCI (Reddy et al., 2022).
- **Fitur teks**: hanya dua field — `query` dan `product_title`. Field `product_description` dan `product_bullet_point` **tidak pernah dibaca maupun dibersihkan** di manapun dalam pipeline ini. Ini murni keputusan ruang lingkup (bukan keterbatasan teknis) agar perbandingan lexical vs semantic terjadi pada representasi teks yang sama persis.
- **Model semantic**: hanya **DeBERTa-v3-base** (He et al., 2021) yang dilaporkan. BERT-base dan RoBERTa-base sempat diuji coba pada tahap eksplorasi awal, namun **dihilangkan dari cakupan pelaporan** sesuai arahan pembimbing, agar penelitian fokus pada satu model semantic yang representatif dibanding menyebar ke banyak varian transformer.
- **Split resmi dataset dipertahankan** — kolom `split` bawaan ESCI dipakai apa adanya, tidak ada pembuatan split acak sendiri yang berisiko mencampur baris antar populasi resmi TRAIN/TEST.
- **Ukuran subset tetap** (fixed, tidak berubah antar-run):
  - 100.000 baris *stratified sample* dari official **TRAIN** → dipecah tetap menjadi **90.000 training** + **10.000 validation**.
  - 30.551 baris *stratified sample* dari official **TEST** → **hanya** dipakai untuk evaluasi akhir (final evaluation), tidak pernah untuk pengembangan model, pemilihan model, tuning hyperparameter, atau *early stopping*.
- **Seed** = `42` di seluruh tahap (sampling, split, training) untuk reproduksibilitas.

## Metrik Evaluasi

- **Primer**: Macro-F1 (rata-rata F1 tiap kelas, tidak dibobot ukuran kelas — penting karena distribusi label ESCI timpang, mayoritas kelas E) dan Weighted-F1.
- **Sekunder**: Accuracy dan Micro-F1.
- **Analisis karakteristik**: F1 per kelas (E/S/C/I) dan confusion matrix, untuk melihat pola kesalahan tiap pendekatan (mis. apakah semantic lebih baik membedakan Substitute vs Complement dibanding lexical), bukan klaim sebab-akibat.

## Alur Notebook Ini

| # | Tahap | Membaca test 30.551? |
|---|---|---|
| 1 | Mount Drive + clone repo | Tidak |
| 2 | Cek GPU | Tidak |
| 3 | Build fixed representative subsets | Tidak (hanya membaca raw data mentah) |
| 4 | Audit ukuran & distribusi label | Tidak |
| 5 | Audit *no-leakage* | Tidak |
| 6 | Development — Lexical (TF-IDF + SVM) | **Tidak** |
| 7 | Development — Semantic (DeBERTa-v3-base) | **Tidak** |
| 8 | Tabel hasil validasi (untuk pemilihan model) | **Tidak** |
| 9 | **Final evaluation** | **Ya — HANYA di sini** |
| 10 | Tabel metrik final | Ya (baca hasil, bukan baca ulang data) |
| 11 | Per-class F1 + confusion matrix | Ya (baca hasil) |
| 12 | Ringkasan & keterbatasan | — |

> **Catatan penting**: Sel di bagian 9 (*Final Evaluation*) sengaja dipisah jauh dari sel development, dan diberi peringatan tebal, supaya tidak ada kemungkinan tidak sengaja mengevaluasi ke test set sebelum semua keputusan pengembangan (hyperparameter, strategi TF-IDF, dsb.) benar-benar dikunci.

---
## 1. Setup Lingkungan: Mount Google Drive + Clone Repository

Sel berikut melakukan tiga hal:
1. Mount Google Drive (tempat data mentah ESCI disimpan dan tempat semua output eksperimen akan ditulis secara persisten, supaya tidak hilang kalau runtime Colab terputus).
2. Clone (atau `pull` kalau sudah pernah di-clone sebelumnya) source code dari repository GitHub — semua logika pemrosesan data, cleaning, TF-IDF, SVM, dan fine-tuning DeBERTa ada di sana, bukan ditulis ulang di notebook, supaya notebook ini murni menjadi *runner* / laporan eksekusi, bukan tempat logika bercampur dengan narasi.
3. Install dependency dari `requirements.txt`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

drive.mount("/content/drive")

# ============================================================
# KONFIGURASI REPOSITORY — sesuaikan kalau kamu push ke repo/branch lain
# ============================================================
REPOSITORY_URL = "https://github.com/niciiu/esci-product-relevance.git"
BRANCH = "main"
COLAB_ROOT = Path("/content/esci-product-relevance")

# ============================================================
# KONFIGURASI FOLDER PROJECT DI GOOGLE DRIVE
# Sesuaikan ke lokasi folder project kamu di Drive.
# Folder ini HARUS berisi subfolder data/raw/ dengan dua file:
#   - shopping_queries_dataset_examples.parquet
#   - shopping_queries_dataset_products.parquet
# ============================================================
DRIVE_FOLDER = Path("/content/drive/MyDrive/esci-product-relevance")

RAW_DATA_DIR = DRIVE_FOLDER / "data" / "raw"
OUTPUT_DIR = DRIVE_FOLDER / "experiment_outputs"

if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Folder raw dataset tidak ditemukan:\n{RAW_DATA_DIR}\n"
        "Pastikan shopping_queries_dataset_examples.parquet dan "
        "shopping_queries_dataset_products.parquet sudah diupload ke Drive di lokasi ini."
    )

if (COLAB_ROOT / ".git").exists():
    print("Repository sudah ada. Melakukan update (git pull)...")
    subprocess.run(["git", "-C", str(COLAB_ROOT), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(COLAB_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(COLAB_ROOT), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", "--branch", BRANCH, REPOSITORY_URL, str(COLAB_ROOT)], check=True)

os.chdir(COLAB_ROOT)
if str(COLAB_ROOT) not in sys.path:
    sys.path.insert(0, str(COLAB_ROOT))

print("Installing requirements...")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"], check=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 70)
print("KONFIGURASI")
print("=" * 70)
print("Repository        :", COLAB_ROOT)
print("Folder Drive       :", DRIVE_FOLDER)
print("Raw data           :", RAW_DATA_DIR)
print("Output persisten   :", OUTPUT_DIR)
print("Commit saat ini    :", subprocess.run(
    ["git", "-C", str(COLAB_ROOT), "log", "-1", "--oneline"],
    capture_output=True, text=True,
).stdout.strip())
print("=" * 70)


---
## 2. Pemeriksaan GPU

Fine-tuning DeBERTa-v3-base pada 90.000 baris jauh lebih cepat dengan GPU dibanding CPU (bisa berbeda dari hitungan menit menjadi berjam-jam). Pastikan runtime Colab sudah diset ke GPU: **Runtime > Change runtime type > Hardware accelerator > GPU**, sebelum menjalankan Bagian 7.

In [ ]:
import torch

print("CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("Memori GPU:", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print(
        "PERINGATAN: tidak ada GPU terdeteksi. Training DeBERTa akan SANGAT lambat di CPU.\n"
        "Buka Runtime > Change runtime type > pilih GPU, lalu jalankan ulang notebook dari awal."
    )


---
## 3. Membangun Fixed Representative Subsets

Tahap ini membaca data mentah ESCI dan menghasilkan **empat file parquet tetap** yang akan dipakai oleh seluruh eksperimen setelahnya. Langkah di dalamnya:

1. **Filter locale + versi** — hanya ambil baris dengan `product_locale == "us"` dan `large_version == 1` (`src/data/load.py::load_us_esci`). Filter ini didorong langsung ke Parquet reader (predicate pushdown), jadi baris locale lain tidak pernah masuk ke memori.
2. **Pisahkan split resmi** — memakai kolom `split` bawaan dataset (`train` / `test`), **bukan** membuat split acak baru (`src/data/split.py::get_official_split`). Ini penting secara metodologis: kolom `split` pada ESCI sudah dirancang oleh penerbit dataset supaya representatif, jadi mempertahankannya menjaga validitas eksternal hasil penelitian.
3. **Stratified sampling terpisah** untuk TRAIN dan TEST (`src/data/sample.py::stratified_sample`) — masing-masing 100.000 dan 30.551 baris, proporsi label E/S/C/I dipertahankan mendekati populasi aslinya (memakai *largest remainder method* supaya total sampel pas 100.000/30.551, bukan dibulatkan sembarangan).
4. **Split tetap training/validation** dari 100.000 TRAIN subset menjadi 90.000/10.000 (`stratified_train_validation_split`), memakai `StratifiedShuffleSplit` dari scikit-learn supaya proporsi label terjaga di kedua sisi.

Karena TRAIN dan TEST subset disampling dari **populasi resmi yang sudah terpisah sejak awal** (kolom `split`), keduanya otomatis disjoint (tidak overlap) — ini akan diverifikasi ulang secara eksplisit di Bagian 5.

Jalankan sel ini **sekali saja** per seed. Kalau file split sudah ada di Drive dari run sebelumnya, sel ini aman dijalankan ulang (idempotent) — ia akan menghasilkan angka yang identik karena seed dikunci ke 42.

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "scripts.build_us_representative_subset",
        "--raw-data-dir", str(RAW_DATA_DIR),
        "--output-dir", str(DRIVE_FOLDER),
        "--seed", "42",
    ],
    check=True,
)


---
## 4. Audit Ukuran Dataset dan Distribusi Label

Tabel-tabel di bawah ini adalah bukti kuantitatif metodologi sampling untuk dilampirkan di Bab 3 (Metodologi) atau Bab 4 (Hasil) proposal:

- Ukuran populasi asli (official TRAIN/TEST) vs. ukuran subset representatif, dalam jumlah baris **dan** persentase terhadap populasi.
- Distribusi label E/S/C/I di setiap tahap sampling — dipakai untuk menunjukkan bahwa proporsi kelas asli (yang timpang, mayoritas *Exact*) berhasil dipertahankan oleh stratified sampling, bukan berubah drastis karena random sampling biasa.

In [ ]:
import json
import pandas as pd

REPORT_PATH = DRIVE_FOLDER / "esci_us_sampling_report.json"
with REPORT_PATH.open(encoding="utf-8") as file:
    report = json.load(file)

print("Metadata dataset")
display(pd.DataFrame([report["dataset"]]))

print("\nMetadata metodologi")
display(pd.DataFrame([report["methodology"]]))

print("\nUkuran dataset dan persentase terhadap populasi")
size_rows = [{"split": name, **values} for name, values in report["size_audit"].items()]
display(pd.DataFrame(size_rows).fillna(""))

print("\nDistribusi label E/S/C/I dalam persen")
distribution_rows = []
for split_name, values in report["split_summaries"].items():
    for label, label_values in values["label_distribution"].items():
        distribution_rows.append({"split": split_name, "label": label, **label_values})
distribution_table = pd.DataFrame(distribution_rows)
display(distribution_table.pivot(index="label", columns="split", values="percentage").round(4))

print("\nDistribusi label E/S/C/I dalam jumlah baris")
display(distribution_table.pivot(index="label", columns="split", values="count"))


---
## 5. Audit *No-Leakage* (Independen dari Kode Training)

Salah satu pertanyaan yang paling sering muncul saat sidang untuk penelitian *machine learning* adalah **"bagaimana Anda memastikan tidak ada kebocoran data (data leakage) antara train, validation, dan test?"**

Sel berikut menjalankan `scripts/sanity_check_no_leakage.py`, sebuah script audit yang **terpisah** dari kode training (jadi bukan "menandai dirinya sendiri lulus") dan memeriksa:

1. **Tidak ada `example_id` yang sama** di antara training/validation/test — pengecekan langsung pada level baris data mentah, bukan asumsi.
2. **Tidak ada pasangan (query, product_title) identik** yang muncul di lebih dari satu split — pengecekan yang lebih ketat dari sekadar `example_id`, karena secara teori baris berbeda bisa saja mengandung teks yang sama persis.
3. **TF-IDF vectorizer menolak `.transform()` sebelum `.fit()`** — bukti kode bahwa alur *fit-hanya-di-training, transform-di-validation/test* benar-benar ditegakkan lewat guard di levelkode, bukan sekadar konvensi penulisan.
4. **Distribusi label tiap split** dicetak ulang sebagai pemeriksaan visual terakhir.

Output sel ini bisa langsung difoto/screenshot untuk lampiran bukti metodologi di proposal.

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "scripts.sanity_check_no_leakage",
        "--data-dir", str(DRIVE_FOLDER),
        "--config", "configs/main_experiment.yaml",
    ],
    check=True,
)


---
## 6. Preprocessing Teks

Preprocessing dijalankan **otomatis di dalam pipeline** (`src/data/clean.py`) setiap kali sebuah eksperimen dijalankan — bukan sebagai cell notebook terpisah. Ini sengaja dirancang begitu supaya preprocessing selalu konsisten dan tidak mungkin ada versi data "yang lupa dibersihkan ulang" antara satu eksperimen dengan eksperimen lain.

Ringkasan preprocessing per pendekatan:

| Tahap | Lexical (TF-IDF + SVM) | Semantic (DeBERTa-v3-base) |
|---|---|---|
| Isi teks kosong | `query`/`product_title` kosong → string kosong `""` | sama |
| HTML & noise | Tag HTML, HTML entity, karakter kontrol, spasi berlebih dibersihkan | sama |
| Casing | **Lowercase** | **Dipertahankan** (huruf besar tetap ada — tokenizer transformer didesain memanfaatkan casing) |
| Tanda baca | **Dihapus** | **Dipertahankan** |
| Stopword | Tidak dihapus (potensi eksperimen lanjutan, di luar cakupan penelitian ini) | Tidak dihapus |

Fitur input yang dipakai **hanya**: `query` dan `product_title`. Baik `product_description` maupun `product_bullet_point` **tidak pernah dibaca** oleh `src/data/clean.py::TEXT_COLUMNS`.

Untuk TF-IDF, vectorizer selalu di-*fit* hanya pada training 90K:
```python
X_train      = vectorizer.fit_transform(train_text)   # fit + transform
X_validation = vectorizer.transform(validation_text)  # transform saja
X_test       = vectorizer.transform(test_text)        # transform saja
```
Sehingga vocabulary dan bobot IDF sepenuhnya berasal dari training, tidak "mengintip" kata-kata yang hanya muncul di validation/test.

---
## 7. Development — Model Lexical (TF-IDF + Linear SVM)

Dua varian yang dibandingkan (didefinisikan di `configs/main_experiment.yaml`):

- **`lexical_separate`**: `query` dan `product_title` masing-masing di-vectorize TF-IDF secara terpisah (vocabulary/IDF sendiri-sendiri), lalu kedua matriks fitur digabung (`hstack`) sebelum masuk ke SVM.
- **`lexical_concatenated`**: `query` dan `product_title` digabung dulu menjadi satu string (dipisah token `[SEP]`), baru di-vectorize TF-IDF bersama sebagai satu vocabulary.

Hyperparameter (dikunci di config, tidak di-tuning saat melihat hasil test):
- `max_features = 20000`, `ngram_range = (1, 2)` (unigram + bigram), `C = 1.0` untuk Linear SVM.

Tahap ini **hanya membaca training 90K dan validation 10K**. Test 30.551 tidak dibaca sama sekali di sini (bisa dicek: argumen `--final-evaluation` tidak disertakan).

In [ ]:
LEXICAL_EXPERIMENTS = ["lexical_separate", "lexical_concatenated"]

for experiment in LEXICAL_EXPERIMENTS:
    print(f"\n{'=' * 70}\nDevelopment: {experiment}\n{'=' * 70}")
    subprocess.run(
        [
            sys.executable, "-m", "scripts.run_main_experiment",
            "--config", "configs/main_experiment.yaml",
            "--data-dir", str(DRIVE_FOLDER),
            "--output-dir", str(OUTPUT_DIR),
            "--experiment", experiment,
        ],
        check=True,
    )


---
## 8. Development — Model Semantic (DeBERTa-v3-base)

Model **microsoft/deberta-v3-base** di-*fine-tune* untuk klasifikasi sekuens berpasangan `(query, product_title)` menjadi 4 kelas ESCI, memakai Hugging Face `Trainer`.

Konfigurasi utama (`configs/main_experiment.yaml`, eksperimen `semantic_deberta`):

| Parameter | Nilai | Alasan |
|---|---|---|
| `max_sequence_length` | 128 token | Query + product title pendek, 128 token lebih dari cukup dan menghemat waktu training dibanding 256/512 |
| `learning_rate` | 2e-5 | Nilai standar untuk fine-tuning encoder transformer |
| `batch_size` | 16 | Disesuaikan untuk GPU single-Colab (T4/A100) |
| `max_epochs` | 10 | Batas atas; jarang tercapai karena early stopping |
| `early_stopping_patience` | 2 | Training dihentikan jika `eval_loss` (dihitung di **validation**, bukan test) tidak membaik selama 2 evaluasi berturut-turut |
| `checkpoint_save_steps` | 500 | Checkpoint disimpan tiap 500 step, persisten di Google Drive |

**Model selection memakai `eval_loss` pada validation**, bukan performa di test — `load_best_model_at_end=True` + `metric_for_best_model="eval_loss"` di `TrainingArguments` (lihat `src/models/semantic_transformer.py`).

**Resume otomatis**: jika runtime Colab terputus di tengah training, jalankan ulang sel ini — training akan otomatis melanjutkan dari checkpoint *lengkap* terakhir (dicek lewat keberadaan `trainer_state.json`, supaya tidak resume dari checkpoint yang korup akibat proses terputus saat menulis file).

Data test **tidak dibaca** pada tahap ini.

In [ ]:
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

subprocess.run(
    [
        sys.executable, "-u", "-m", "scripts.run_main_experiment",
        "--config", "configs/main_experiment.yaml",
        "--data-dir", str(DRIVE_FOLDER),
        "--output-dir", str(OUTPUT_DIR),
        "--experiment", "semantic_deberta",
    ],
    check=True, env=env,
)


---
## 9. Tabel Hasil Validasi (Model Selection — BUKAN Angka Final)

Tabel ini dipakai untuk **membandingkan** ketiga pendekatan selama development, berdasarkan performa di **validation 10K**. Ini adalah dasar untuk menentukan pendekatan mana yang paling menjanjikan — bukan angka yang dilaporkan sebagai hasil akhir penelitian (angka final ada di Bagian 10, setelah final evaluation).

Metrik yang dibandingkan: Macro-F1 (utama), Weighted-F1, Accuracy, Micro-F1.

In [ ]:
ALL_EXPERIMENTS = ["lexical_separate", "lexical_concatenated", "semantic_deberta"]

validation_rows = []
for experiment in ALL_EXPERIMENTS:
    result_path = OUTPUT_DIR / experiment / "results.json"
    with result_path.open(encoding="utf-8") as file:
        result = json.load(file)
    validation_rows.append({"experiment": experiment, **result["validation"]})

validation_results = pd.DataFrame(validation_rows)
display(
    validation_results[["experiment", "macro_f1", "weighted_f1", "accuracy", "micro_f1"]]
    .sort_values("macro_f1", ascending=False)
)


---
## 9.a Grafik Loss Training DeBERTa (Diagnostik)

Plot `training_loss` dan `eval_loss` DeBERTa sepanjang step, untuk melihat apakah model overfitting (eval_loss naik lagi setelah titik tertentu — yang seharusnya sudah ditangkap oleh early stopping) atau masih *underfit* (loss belum stabil saat training berhenti).

In [ ]:
semantic_result_path = OUTPUT_DIR / "semantic_deberta" / "results.json"
with semantic_result_path.open(encoding="utf-8") as file:
    semantic_result = json.load(file)

loss_history = pd.DataFrame(semantic_result["loss_history"])
train_loss = loss_history.dropna(subset=["loss"]) if "loss" in loss_history.columns else pd.DataFrame()
eval_loss = loss_history.dropna(subset=["eval_loss"]) if "eval_loss" in loss_history.columns else pd.DataFrame()

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
if not train_loss.empty:
    plt.plot(train_loss["step"], train_loss["loss"], label="training_loss")
if not eval_loss.empty:
    plt.plot(eval_loss["step"], eval_loss["eval_loss"], label="eval_loss (validation)", marker="o")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("DeBERTa-v3-base: training loss vs validation loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


---
## 10. Final Evaluation

> ### ⚠️ JALANKAN BAGIAN INI HANYA SETELAH SEMUA KEPUTUSAN DEVELOPMENT DIKUNCI
>
> Ini adalah **satu-satunya tahap** dalam seluruh notebook yang membaca 30.551 baris official TEST subset (lihat argumen `--final-evaluation`). Setelah sel ini dijalankan dan hasilnya dilihat, **tidak boleh lagi** ada perubahan hyperparameter, strategi TF-IDF, arsitektur, atau keputusan pengembangan lain yang "disesuaikan" berdasarkan angka test — perilaku semacam itu (*test set leakage melalui iterasi manusia*, bukan leakage teknis) akan membuat metrik akhir menjadi optimis-bias dan tidak valid secara metodologis.
>
> Jika kamu perlu mengubah sesuatu setelah melihat hasil bagian ini, catat secara eksplisit di laporan bahwa keputusan tersebut diambil setelah melihat test, dan idealnya jalankan final evaluation lagi sebagai run yang terpisah/ditandai.

In [ ]:
for experiment in ALL_EXPERIMENTS:
    print(f"\n{'=' * 70}\nFinal evaluation: {experiment}\n{'=' * 70}")
    subprocess.run(
        [
            sys.executable, "-m", "scripts.run_main_experiment",
            "--config", "configs/main_experiment.yaml",
            "--data-dir", str(DRIVE_FOLDER),
            "--output-dir", str(OUTPUT_DIR),
            "--experiment", experiment,
            "--final-evaluation",
        ],
        check=True,
    )


---
## 11. Tabel Metrik Final

Metrik berikut berasal **eksklusif** dari 30.551 baris representative official TEST — ini adalah angka yang dilaporkan sebagai hasil utama penelitian (Bab 4).

In [ ]:
final_rows = []
for experiment in ALL_EXPERIMENTS:
    result_path = OUTPUT_DIR / experiment / "results.json"
    with result_path.open(encoding="utf-8") as file:
        result = json.load(file)
    if not result["final_evaluation_ran"]:
        raise RuntimeError(f"Final evaluation belum dijalankan: {experiment}")
    final_rows.append({"experiment": experiment, **result["final_test"]})

final_results = pd.DataFrame(final_rows)
display(
    final_results[["experiment", "macro_f1", "weighted_f1", "accuracy", "micro_f1"]]
    .sort_values("macro_f1", ascending=False)
)


---
## 12. Analisis Karakteristik: F1 per Kelas dan Confusion Matrix

Bagian ini membandingkan **pola kesalahan** (bukan hanya skor agregat) antar pendekatan — inti dari kata "Karakteristik Hasil Klasifikasi" di judul skripsi.

Yang perlu diperhatikan saat membaca tabel di bawah:
- Kelas **E** (Exact) biasanya punya F1 tertinggi di semua pendekatan karena proporsinya paling besar di data (mayoritas).
- Kelas **C** (Complement) dan **S** (Substitute) sering menjadi yang paling sulit dibedakan satu sama lain — perhatikan sel confusion matrix `actual_S -> pred_C` dan `actual_C -> pred_S`.
- Bandingkan apakah DeBERTa memperbaiki F1 kelas minoritas (S, C, I) dibanding lexical, atau justru performanya didorong oleh kelas mayoritas (E) yang sama-sama mudah bagi kedua pendekatan.

Interpretasi ini adalah **perbandingan performa yang diamati**, bukan klaim sebab-akibat kausal.

In [ ]:
for experiment in ALL_EXPERIMENTS:
    result_path = OUTPUT_DIR / experiment / "results.json"
    with result_path.open(encoding="utf-8") as file:
        result = json.load(file)
    final_test = result["final_test"]

    print(f"\n{'=' * 70}\nExperiment: {experiment}\n{'=' * 70}")

    per_class_f1 = pd.DataFrame(
        [{"label": label, "f1_score": score} for label, score in final_test["per_class_f1"].items()]
    )
    display(per_class_f1)

    confusion = pd.DataFrame(
        final_test["confusion_matrix"],
        index=["actual_E", "actual_S", "actual_C", "actual_I"],
        columns=["pred_E", "pred_S", "pred_C", "pred_I"],
    )
    display(confusion)


---
## 12.a Visualisasi Confusion Matrix (Heatmap)

Versi visual dari tabel confusion matrix di atas — memudahkan audiens sidang melihat pola kesalahan secara sekilas dibanding membaca angka mentah.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, len(ALL_EXPERIMENTS), figsize=(6 * len(ALL_EXPERIMENTS), 5))
if len(ALL_EXPERIMENTS) == 1:
    axes = [axes]

labels = ["E", "S", "C", "I"]
for ax, experiment in zip(axes, ALL_EXPERIMENTS):
    result_path = OUTPUT_DIR / experiment / "results.json"
    with result_path.open(encoding="utf-8") as file:
        result = json.load(file)
    matrix = np.array(result["final_test"]["confusion_matrix"])
    row_normalized = matrix / matrix.sum(axis=1, keepdims=True)

    im = ax.imshow(row_normalized, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(4)); ax.set_xticklabels(labels)
    ax.set_yticks(range(4)); ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(experiment)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{matrix[i, j]}\n({row_normalized[i, j]:.0%})",
                     ha="center", va="center",
                     color="white" if row_normalized[i, j] > 0.5 else "black", fontsize=9)

fig.colorbar(im, ax=axes, shrink=0.7, label="proporsi baris (recall per kelas)")
plt.show()


---
## 13. Ringkasan dan Keterbatasan Penelitian

### Ringkasan (isi setelah eksperimen selesai dijalankan)
- Pendekatan dengan Macro-F1 tertinggi di final test: **[isi setelah Bagian 11 dijalankan]**
- Pola kesalahan yang menonjol dari confusion matrix: **[isi berdasarkan Bagian 12]**
- Apakah hasil ini konsisten dengan hasil di tahap validation (Bagian 9), atau ada pendekatan yang tampak baik di validation tapi turun di test: **[isi]**

### Keterbatasan Penelitian
- Hanya mencakup locale **US** (`product_locale == "us"`) — generalisasi ke locale lain (JP, ES, dll.) tidak diklaim.
- Hanya menggunakan **product_title**, tidak menggunakan `product_description` maupun `product_bullet_point` yang sebenarnya tersedia di dataset — trade-off yang diambil demi kesetaraan perbandingan lexical vs semantic pada representasi teks yang identik.
- Model semantic dibatasi pada **satu backbone** (DeBERTa-v3-base); tidak ada ablasi backbone lain (BERT, RoBERTa) dalam laporan akhir meskipun sempat dieksplorasi di tahap awal.
- Ukuran subset (100K/30.551) adalah representative sample dari populasi resmi yang jauh lebih besar (>1,3 juta baris TRAIN) — dipilih karena keterbatasan waktu komputasi, bukan karena populasi penuh dianggap tidak relevan.
- Tidak dilakukan pengujian signifikansi statistik antar pendekatan (mis. bootstrap confidence interval atau uji McNemar) pada versi ini — bisa menjadi rekomendasi pengembangan lanjutan.

### Rekomendasi Pengembangan Lanjutan
- Menambahkan uji signifikansi statistik antara Macro-F1 lexical vs semantic.
- Mengeksplorasi kombinasi fitur (mis. menambahkan `product_description` sebagai eksperimen tambahan, bukan pengganti eksperimen utama).
- Menguji generalisasi lintas locale (mis. melatih di US, mengevaluasi di locale lain sebagai zero-shot cross-lingual test).